In [ ]:
# ================================================================
# 1D-CNN Shallow Neural Detector
# ================================================================
# Architecture: Embedding → 1D Conv (multi-filter) → Global Max Pool
#               → Dense → Sigmoid
# < 5M parameters — bridges handcrafted features (Stage 2A) and
# full transformer fine-tuning (Stage 2B).
# Research focus:
#   - Does shallow learned detection beat handcrafted features?
#   - Degradation curve vs humanisation level
#   - Robustness to global paraphrasing (conv bias toward local n-grams)
# ================================================================

# ── Cell 1: Install & Imports ──────────────────────────────────
!pip install -q torch torchtext scikit-learn matplotlib seaborn tqdm

import os, json, pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, log_loss,
    accuracy_score, roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings; warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

RESULTS_DIR = "./results/cnn"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Cell 2: Config ─────────────────────────────────────────────
class CNNConfig:
    # Tokenisation
    MAX_VOCAB_SIZE  = 30_000
    MAX_SEQ_LENGTH  = 256       # Local patterns; shorter than transformers
    MIN_WORD_FREQ   = 2

    # Embedding
    EMBED_DIM       = 128       # Intentionally small — <5M param budget

    # Convolution: multi-filter captures unigrams, bigrams, trigrams, 4-grams
    FILTER_SIZES    = [2, 3, 4, 5]
    NUM_FILTERS     = 128       # Per filter size

    # Dense head
    DROPOUT         = 0.4       # Heavier dropout for shallow network
    HIDDEN_DIM      = 256

    # Training
    BATCH_SIZE      = 64
    EPOCHS          = 10
    LR              = 1e-3
    WEIGHT_DECAY    = 1e-4
    PATIENCE        = 3         # Early stopping patience

    # Data
    VAL_SPLIT       = 0.1
    SEED            = 42

    @property
    def total_filter_dim(self):
        return len(self.FILTER_SIZES) * self.NUM_FILTERS

cfg = CNNConfig()

# ── Cell 3: Vocabulary Builder ─────────────────────────────────

class Vocabulary:
    PAD = "<PAD>"; UNK = "<UNK>"

    def __init__(self, max_size=30_000, min_freq=2):
        self.max_size = max_size
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}

    def build(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(str(text).lower().split())

        # Keep PAD at 0, UNK at 1
        vocab = [w for w, f in counter.most_common(self.max_size)
                 if f >= self.min_freq]
        special = [self.PAD, self.UNK]
        all_tokens = special + vocab

        self.word2idx = {w: i for i, w in enumerate(all_tokens)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        print(f"  Vocabulary size: {len(self.word2idx):,}")
        return self

    def encode(self, text, max_length):
        tokens = str(text).lower().split()[:max_length]
        ids = [self.word2idx.get(t, 1) for t in tokens]   # 1 = UNK
        # Pad / truncate to max_length
        ids = ids + [0] * (max_length - len(ids))         # 0 = PAD
        return ids


# ── Cell 4: Dataset Class ──────────────────────────────────────

class CNNTextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length):
        self.encodings = [vocab.encode(t, max_length) for t in texts]
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encodings[idx], dtype=torch.long),
            torch.tensor(self.labels[idx],    dtype=torch.float),
        )


# ── Cell 5: Model Architecture ─────────────────────────────────

class MultiFilterCNN(nn.Module):
    """
    Embedding → N parallel 1D Conv layers (different kernel sizes)
    → Global Max Pool per conv → Concatenate → Dropout → Dense → Sigmoid

    Parameter count target: < 5M
    """

    def __init__(self, vocab_size, embed_dim, filter_sizes,
                 num_filters, hidden_dim, dropout):
        super().__init__()

        # Embedding (shared)
        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0)

        # Parallel convolutional branches
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(
                    in_channels  = embed_dim,
                    out_channels = num_filters,
                    kernel_size  = ks,
                    padding      = ks // 2,
                ),
                nn.BatchNorm1d(num_filters),
                nn.ReLU(),
            )
            for ks in filter_sizes
        ])

        total_filters = len(filter_sizes) * num_filters

        # Dense classification head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(total_filters, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # x: (B, L)
        emb = self.embedding(x)            # (B, L, E)
        emb = emb.permute(0, 2, 1)        # (B, E, L) — Conv1d expects (B, C, L)

        # Parallel convolutions + global max pool
        pooled = []
        for conv in self.convs:
            out  = conv(emb)               # (B, F, L')
            pool = F.max_pool1d(out, out.size(2)).squeeze(2)  # (B, F)
            pooled.append(pool)

        cat = torch.cat(pooled, dim=1)     # (B, total_filters)
        logit = self.classifier(cat)       # (B, 1)
        return logit.squeeze(1)            # (B,)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── Cell 6: Load Data ──────────────────────────────────────────
def encode_labels(series):
    return (series == "llm").astype(int).tolist()

hc3_train  = pd.read_csv("hc3_train.csv")
hc3_test   = pd.read_csv("hc3_test.csv")
eli5_train = pd.read_csv("eli5_train.csv")
eli5_test  = pd.read_csv("eli5_test.csv")

# Validation splits
hc3_tr, hc3_val   = train_test_split(hc3_train,  test_size=cfg.VAL_SPLIT,
                                      random_state=cfg.SEED, stratify=hc3_train["label"])
eli5_tr, eli5_val = train_test_split(eli5_train, test_size=cfg.VAL_SPLIT,
                                      random_state=cfg.SEED, stratify=eli5_train["label"])

print(f"HC3  Train={len(hc3_tr):,} Val={len(hc3_val):,} Test={len(hc3_test):,}")
print(f"ELI5 Train={len(eli5_tr):,} Val={len(eli5_val):,} Test={len(eli5_test):,}")


# ── Cell 7: Training Loop ──────────────────────────────────────

def train_cnn(train_df, val_df, test_data_dict, tag=""):
    """
    Full training loop with early stopping.
    Returns: trained model, tokenizer vocab, results dict.
    """
    print(f"\n{'='*60}")
    print(f"Training 1D-CNN  [{tag}]")
    print(f"{'='*60}")

    # Build vocabulary on training data
    vocab = Vocabulary(cfg.MAX_VOCAB_SIZE, cfg.MIN_WORD_FREQ)
    vocab.build(train_df["text"].tolist())

    # Build datasets
    tr_ds  = CNNTextDataset(train_df["text"], encode_labels(train_df["label"]), vocab, cfg.MAX_SEQ_LENGTH)
    val_ds = CNNTextDataset(val_df["text"],   encode_labels(val_df["label"]),   vocab, cfg.MAX_SEQ_LENGTH)

    tr_loader  = DataLoader(tr_ds,  batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

    # Build model
    model = MultiFilterCNN(
        vocab_size   = len(vocab.word2idx),
        embed_dim    = cfg.EMBED_DIM,
        filter_sizes = cfg.FILTER_SIZES,
        num_filters  = cfg.NUM_FILTERS,
        hidden_dim   = cfg.HIDDEN_DIM,
        dropout      = cfg.DROPOUT,
    ).to(device)

    n_params = model.count_parameters()
    print(f"  Parameters : {n_params:,}  ({'<5M ✅' if n_params < 5_000_000 else '>5M ⚠️'})")

    criterion = nn.BCEWithLogitsLoss()
    optimizer = Adam(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=1, factor=0.5)

    best_val_auc = 0.0
    best_state   = None
    patience_ctr = 0
    history      = []

    for epoch in range(1, cfg.EPOCHS + 1):
        # ── Train ──
        model.train()
        total_loss = 0.0
        for x, y in tqdm(tr_loader, desc=f"Epoch {epoch}", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss   = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        # ── Validate ──
        model.eval()
        val_scores, val_labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                logits = model(x.to(device))
                probs  = torch.sigmoid(logits).cpu().numpy()
                val_scores.extend(probs)
                val_labels.extend(y.numpy())

        val_auc = roc_auc_score(val_labels, val_scores)
        scheduler.step(val_auc)

        avg_loss = total_loss / len(tr_loader)
        print(f"  Epoch {epoch:2d} | Loss={avg_loss:.4f} | Val AUC={val_auc:.4f}")
        history.append({"epoch": epoch, "loss": avg_loss, "val_auc": val_auc})

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= cfg.PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    # Restore best
    model.load_state_dict(best_state)
    model.eval()

    # ── Evaluate on test sets ──
    results = {}

    for test_name, test_df in test_data_dict.items():
        te_ds = CNNTextDataset(test_df["text"], encode_labels(test_df["label"]),
                               vocab, cfg.MAX_SEQ_LENGTH)
        te_loader = DataLoader(te_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

        scores, labels = [], []
        with torch.no_grad():
            for x, y in te_loader:
                probs = torch.sigmoid(model(x.to(device))).cpu().numpy()
                scores.extend(probs)
                labels.extend(y.numpy())

        scores = np.array(scores)
        labels = np.array(labels)
        preds  = (scores > 0.5).astype(int)

        results[test_name] = {
            "y_true":               labels,
            "y_pred":               preds,
            "detectability_scores": scores,
            "roc_auc":              roc_auc_score(labels, scores),
            "brier_score":          brier_score_loss(labels, scores),
            "log_loss":             log_loss(labels, scores),
            "accuracy":             accuracy_score(labels, preds),
        }
        print(f"  {test_name:20s} → AUC={results[test_name]['roc_auc']:.4f}  "
              f"Acc={results[test_name]['accuracy']:.4f}")

    return model, vocab, results, history


# ── Cell 8: Train on HC3 ───────────────────────────────────────
model_hc3, vocab_hc3, cnn_hc3_results, history_hc3 = train_cnn(
    hc3_tr, hc3_val,
    {"hc3_to_hc3": hc3_test, "hc3_to_eli5": eli5_test},
    tag="HC3",
)

# ── Cell 9: Train on ELI5 ─────────────────────────────────────
model_eli5, vocab_eli5, cnn_eli5_results, history_eli5 = train_cnn(
    eli5_tr, eli5_val,
    {"eli5_to_eli5": eli5_test, "eli5_to_hc3": hc3_test},
    tag="ELI5",
)

cnn_results = {"1D-CNN": {**cnn_hc3_results, **cnn_eli5_results}}

# ── Cell 10: Training Curves ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, history, tag in zip(axes,
                             [history_hc3, history_eli5],
                             ["HC3", "ELI5"]):
    epochs = [h["epoch"]   for h in history]
    losses = [h["loss"]    for h in history]
    aucs   = [h["val_auc"] for h in history]
    ax.plot(epochs, losses, "-o", label="Train Loss", color="#e74c3c")
    ax2 = ax.twinx()
    ax2.plot(epochs, aucs, "-s", label="Val AUC", color="#2ecc71")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax2.set_ylabel("Val AUC")
    ax.set_title(f"1D-CNN Training Curve — {tag}")
    ax.legend(loc="upper left"); ax2.legend(loc="upper right")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cnn_training_curves.png", dpi=150)
plt.show()

# ── Cell 11: Score Distributions ──────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("1D-CNN: Detectability Score Distributions", fontsize=16, fontweight="bold")

for j, (eval_name, d) in enumerate(cnn_results["1D-CNN"].items()):
    ax = axes[j]
    ax.hist(d["detectability_scores"][d["y_true"]==0], bins=30, alpha=0.6,
            label="Human", color="#3498db", range=(0,1))
    ax.hist(d["detectability_scores"][d["y_true"]==1], bins=30, alpha=0.6,
            label="LLM",   color="#e74c3c", range=(0,1))
    ax.set_title(f"{eval_name}\nAUC={d['roc_auc']:.3f}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Detectability Score"); ax.set_ylabel("Frequency")
    ax.axvline(0.5, color="k", linestyle="--", alpha=0.4); ax.legend(); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cnn_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Cell 12: Degradation Curve (Humanisation Simulation) ──────
# Simulate varying levels of "humanisation" by injecting human text
# tokens into LLM text at increasing ratios. Measures how gracefully
# the CNN degrades vs how abruptly transformer-based models collapse.

print("\n" + "="*70)
print("DEGRADATION ANALYSIS — CNN Robustness to Mixing")
print("="*70)

def interpolate_texts(llm_texts, human_texts, mix_ratio):
    """
    Return texts where mix_ratio fraction of tokens come from human text.
    0.0 = pure LLM, 1.0 = pure human.
    """
    mixed = []
    for llm, human in zip(llm_texts, human_texts):
        llm_toks   = str(llm).split()
        human_toks = str(human).split()
        n_total    = len(llm_toks)
        n_human    = int(n_total * mix_ratio)
        n_llm      = n_total - n_human
        combined   = llm_toks[:n_llm] + human_toks[:n_human]
        mixed.append(" ".join(combined))
    return mixed


# Build test pairs
hc3_llm   = hc3_test[hc3_test["label"]=="llm"]["text"].reset_index(drop=True)
hc3_human = hc3_test[hc3_test["label"]=="human"]["text"].reset_index(drop=True)
n_pairs   = min(len(hc3_llm), len(hc3_human))
hc3_llm   = hc3_llm[:n_pairs]
hc3_human = hc3_human[:n_pairs]

mix_ratios = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
degrade_results = {}

for ratio in mix_ratios:
    mixed_texts = interpolate_texts(hc3_llm, hc3_human, ratio)
    labels_arr  = [1] * n_pairs   # still labelled as LLM-origin

    ds = CNNTextDataset(mixed_texts, labels_arr, vocab_hc3, cfg.MAX_SEQ_LENGTH)
    loader = DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

    scores = []
    model_hc3.eval()
    with torch.no_grad():
        for x, _ in loader:
            probs = torch.sigmoid(model_hc3(x.to(device))).cpu().numpy()
            scores.extend(probs)

    avg_score = float(np.mean(scores))
    degrade_results[ratio] = avg_score

ratios  = list(degrade_results.keys())
scores_ = list(degrade_results.values())

plt.figure(figsize=(8, 5))
plt.plot(ratios, scores_, "-o", linewidth=2.5, color="#9b59b6", markersize=8)
plt.axhline(0.5, color="k", linestyle="--", alpha=0.5, label="Decision boundary")
plt.xlabel("Human token mix ratio (0=pure LLM, 1=pure human)")
plt.ylabel("Mean CNN Detectability Score (LLM probability)")
plt.title("1D-CNN Degradation Curve\n(How gracefully does detection degrade as text is humanised?)")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cnn_degradation_curve.png", dpi=150)
plt.show()

print("Degradation curve data:")
for r, s in zip(ratios, scores_):
    print(f"  Mix ratio {r:.1f} → CNN score = {s:.4f}")

# ── Cell 13: Filter Visualisation (Top-Activated N-Grams) ─────
print("\n" + "="*70)
print("TOP N-GRAM PATTERNS PER FILTER")
print("="*70)

def get_top_ngrams_for_filter(model, vocab, texts, filter_idx, filter_size,
                               top_k=10, n_examples=500):
    """Find input n-grams that maximally activate a specific filter."""
    idx2word = vocab.idx2word
    model.eval()

    activation_records = []

    for text in texts[:n_examples]:
        tokens = str(text).lower().split()[:cfg.MAX_SEQ_LENGTH]
        ids    = [vocab.word2idx.get(t, 1) for t in tokens]
        ids    += [0] * (cfg.MAX_SEQ_LENGTH - len(ids))
        x      = torch.tensor([ids], dtype=torch.long).to(device)

        # Hook to capture conv output
        with torch.no_grad():
            emb  = model.embedding(x).permute(0, 2, 1)
            conv_branch = model.convs[filter_idx - (filter_size - min(cfg.FILTER_SIZES))]
            # Find the right conv by filter size
            for cb in model.convs:
                if cb[0].kernel_size[0] == filter_size:
                    out = cb(emb).squeeze(0)  # (F, L)
                    break
            # Max activation position for this filter
            max_val, max_pos = out[filter_idx % cfg.NUM_FILTERS].max(0)
            pos = max_pos.item()
            ngram = tokens[max(0, pos - filter_size//2):
                           max(0, pos - filter_size//2) + filter_size]
            activation_records.append((max_val.item(), " ".join(ngram)))

    activation_records.sort(reverse=True)
    return activation_records[:top_k]

# Show top-activated bigrams and trigrams for first few filters
sample_texts = hc3_test["text"].tolist()[:200]
for fs in [2, 3]:
    print(f"\nFilter size {fs} — Top 10 activating n-grams (filter 0):")
    ngrams = get_top_ngrams_for_filter(model_hc3, vocab_hc3, sample_texts,
                                        filter_idx=0, filter_size=fs)
    for val, ng in ngrams:
        print(f"    [{val:6.2f}] '{ng}'")

# ── Cell 14: Performance Summary ──────────────────────────────
rows = []
for eval_name, d in cnn_results["1D-CNN"].items():
    rows.append({
        "Detector":        "1D-CNN",
        "Evaluation":      eval_name,
        "ROC-AUC":         d["roc_auc"],
        "Accuracy":        d["accuracy"],
        "Brier Score":     d["brier_score"],
        "Log Loss":        d["log_loss"],
        "Mean Human Score":d["detectability_scores"][d["y_true"]==0].mean(),
        "Mean LLM Score":  d["detectability_scores"][d["y_true"]==1].mean(),
    })

summary_df = pd.DataFrame(rows)
print("\n" + "="*70)
print("1D-CNN PERFORMANCE SUMMARY")
print("="*70)
print(summary_df.round(4).to_string(index=False))
summary_df.to_csv(f"{RESULTS_DIR}/cnn_performance_summary.csv", index=False)

with open(f"{RESULTS_DIR}/cnn_results.pkl", "wb") as f:
    pickle.dump(cnn_results, f)

print(f"\n✅ All results saved to {RESULTS_DIR}/")
print("🎯 1D-CNN Shallow Neural Detector complete.")